# SBER AI Journey — 3D Mesh Quality Control v2.1: Analysis & Visualization**Competition Metric:** `f1_final = 10 x F1(quality) + 10 x F1_weighted(defects)`This notebook provides comprehensive analysis of model predictions, thresholds, per-class performance, and interpretability visualizations (Grad-CAM, attention weights).

## 1. Setup & Load Results

In [ ]:
import sys, os, jsonimport numpy as npimport pandas as pdimport matplotlibmatplotlib.use('Agg')import matplotlib.pyplot as pltimport seaborn as snsplt.style.use('seaborn-v0_8-whitegrid')plt.rcParams['figure.dpi'] = 120plt.rcParams['font.size'] = 11SOLUTION_DIR = os.path.join(os.getcwd(), 'solution')LOG_DIR = os.path.join(os.getcwd(), 'logs')sys.path.insert(0, SOLUTION_DIR)DEFECT_COLS = [    "abstract", "artifacts", "intersection", "lowpoly",    "noisy", "open", "partial", "scale", "set", "simple"]print("Libraries loaded successfully.")

In [ ]:
# Load CV resultscv_path = os.path.join(LOG_DIR, 'cv_results.json')if os.path.exists(cv_path):    with open(cv_path) as f:        cv_results = json.load(f)    print(f"Loaded CV results from {cv_path}")    print(f"Folds: {len(cv_results.get('fold_results', []))}")    avg = cv_results['avg_metrics']    print(f"\n  F1_final:   {avg['f1_final_mean']:.2f} +/- {avg['f1_final_std']:.2f}")    print(f"  F1_quality: {avg['f1_quality_mean']:.4f} +/- {avg['f1_quality_std']:.4f}")    print(f"  F1_defects: {avg['f1_defects_mean']:.4f} +/- {avg['f1_defects_std']:.4f}")    print(f"  Temperature: {cv_results.get('avg_temperature', 1.0):.3f}")else:    print(f"CV results not found at {cv_path}")    print("Run training first to generate results.")    cv_results = None

## 2. Training Curves

In [ ]:
if cv_results and cv_results.get('fold_results'):    fig, axes = plt.subplots(1, 3, figsize=(18, 5))        for fold_res in cv_results['fold_results']:        h = fold_res['history']        epochs = range(1, len(h['train_loss']) + 1)        fold = fold_res['fold']                axes[0].plot(epochs, h['train_loss'], alpha=0.7, label=f'Fold {fold+1}')        axes[1].plot(epochs, h['val_f1_final'], alpha=0.7, label=f'Fold {fold+1}')        axes[2].plot(epochs, h['lr'], alpha=0.7, label=f'Fold {fold+1}')        axes[0].set_title('Training Loss')    axes[0].set_xlabel('Epoch')    axes[0].set_ylabel('Loss')    axes[0].legend(fontsize=9)    axes[0].grid(True, alpha=0.3)        axes[1].set_title('Validation F1_final')    axes[1].set_xlabel('Epoch')    axes[1].set_ylabel('F1_final')    axes[1].legend(fontsize=9)    axes[1].grid(True, alpha=0.3)        axes[2].set_title('Learning Rate')    axes[2].set_xlabel('Epoch')    axes[2].set_ylabel('LR')    axes[2].legend(fontsize=9)    axes[2].grid(True, alpha=0.3)        plt.tight_layout()    plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')    plt.show()    print("Training curves saved.")else:    print("No CV results available.")

## 3. Submission Analysis

In [ ]:
sub_path = os.path.join(os.getcwd(), 'submission.csv')proba_path = sub_path.replace('.csv', '_proba.csv')if os.path.exists(sub_path):    sub = pd.read_csv(sub_path)    print(f"Submission: {sub.shape[0]} samples, {sub.shape[1]} columns")    print(f"\nQuality distribution:")    print(sub['quality'].value_counts().to_string())    print(f"\nPer-class positive rate:")    for col in DEFECT_COLS:        rate = sub[col].mean() * 100        print(f"  {col:15s}: {rate:5.1f}%")        if os.path.exists(proba_path):        proba = pd.read_csv(proba_path)        fig, axes = plt.subplots(2, 2, figsize=(16, 12))                # Probability distributions        for i, col in enumerate(DEFECT_COLS):            axes[0, 0].hist(proba[col], bins=50, alpha=0.6, label=col)        axes[0, 0].set_title('Probability Distributions (all classes)')        axes[0, 0].set_xlabel('Probability')        axes[0, 0].set_ylabel('Count')        axes[0, 0].legend(fontsize=8)        axes[0, 0].grid(True, alpha=0.3)                # Per-class positive rate bar        rates = sub[DEFECT_COLS].mean() * 100        colors = ['#e74c3c' if r > 5 else '#3498db' for r in rates]        axes[0, 1].barh(DEFECT_COLS, rates, color=colors)        axes[0, 1].set_title('Predicted Positive Rate (%)')        axes[0, 1].grid(True, alpha=0.3, axis='x')                # Probability heatmap (sample)        sample = proba[DEFECT_COLS].iloc[:min(200, len(proba))]        corr = sample.corr()        sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlBu_r',                     center=0, ax=axes[1, 0], cbar_kws={'shrink': 0.8},                    xticklabels=[c[:6] for c in DEFECT_COLS],                    yticklabels=[c[:6] for c in DEFECT_COLS])        axes[1, 0].set_title('Defect Correlation Matrix (sample)')                # Mean probability per class        mean_proba = proba[DEFECT_COLS].mean()        std_proba = proba[DEFECT_COLS].std()        x = range(len(DEFECT_COLS))        axes[1, 1].bar(x, mean_proba, yerr=std_proba, color='steelblue', alpha=0.7)        axes[1, 1].set_xticks(x)        axes[1, 1].set_xticklabels([c[:6] for c in DEFECT_COLS], rotation=45, ha='right')        axes[1, 1].set_title('Mean Probability +/- Std')        axes[1, 1].set_ylabel('Probability')        axes[1, 1].grid(True, alpha=0.3)                plt.tight_layout()        plt.savefig('submission_analysis.png', dpi=150, bbox_inches='tight')        plt.show()else:    print("No submission.csv found. Run inference first.")

## 4. Threshold Analysis (v2.1)

In [ ]:
if cv_results and cv_results.get('fold_results'):    fig, axes = plt.subplots(1, 2, figsize=(16, 6))        # Per-fold thresholds    thresholds = np.array([f['thresholds'] for f in cv_results['fold_results']])    avg_thresh = thresholds.mean(axis=0)        x = np.arange(len(DEFECT_COLS))    for fold_i in range(len(thresholds)):        axes[0].plot(x, thresholds[fold_i], 'o-', alpha=0.4, markersize=4)    axes[0].plot(x, avg_thresh, 's-', color='black', markersize=8, linewidth=2, label='Average')    axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Default 0.5')    axes[0].set_xticks(x)    axes[0].set_xticklabels([c[:6] for c in DEFECT_COLS], rotation=45, ha='right')    axes[0].set_ylabel('Threshold')    axes[0].set_title('Per-Fold Optimized Thresholds (v2.1)')    axes[0].legend()    axes[0].grid(True, alpha=0.3)        # Temperature per fold    temps = [f.get('temperature', 1.0) for f in cv_results['fold_results']]    axes[1].bar(range(len(temps)), temps, color=['#2ecc71' if t == 1.0 else '#e67e22' for t in temps])    axes[1].axhline(1.0, color='red', linestyle='--', alpha=0.5)    axes[1].set_xticks(range(len(temps)))    axes[1].set_xticklabels([f'Fold {i+1}' for i in range(len(temps))])    axes[1].set_ylabel('Temperature T')    axes[1].set_title('Learned Temperature per Fold')    axes[1].grid(True, alpha=0.3, axis='y')        print("Threshold & Temperature analysis:")    for i, col in enumerate(DEFECT_COLS):        print(f"  {col:15s}: threshold={avg_thresh[i]:.3f} (default=0.500)")    print(f"\n  Average temperature: {np.mean(temps):.3f}")        plt.tight_layout()    plt.savefig('threshold_analysis.png', dpi=150, bbox_inches='tight')    plt.show()else:    print("No CV results available.")

## 5. Per-Fold Metrics Comparison

In [ ]:
if cv_results and cv_results.get('fold_results'):    folds = cv_results['fold_results']    metrics_names = ['f1_quality', 'f1_defects', 'f1_final']        fig, axes = plt.subplots(1, 3, figsize=(18, 5))        for ax, metric in zip(axes, metrics_names):        vals = [f['final_metrics'][metric] for f in folds]        mean_val = np.mean(vals)        std_val = np.std(vals)                bars = ax.bar(range(len(vals)), vals, color='steelblue', alpha=0.7)        ax.axhline(mean_val, color='red', linestyle='--', linewidth=2,                     label=f'Mean={mean_val:.4f}')        ax.fill_between(range(len(vals))-0.4, range(len(vals))+0.4,                         mean_val-std_val, mean_val+std_val, alpha=0.1, color='red')                ax.set_xticks(range(len(vals)))        ax.set_xticklabels([f'Fold {f["fold"]+1}' for f in folds])        ax.set_title(f'{metric}')        ax.set_ylabel('Score')        ax.legend()        ax.grid(True, alpha=0.3, axis='y')        plt.tight_layout()    plt.savefig('per_fold_metrics.png', dpi=150, bbox_inches='tight')    plt.show()        print("Per-fold summary:")    for f in folds:        m = f['final_metrics']        print(f"  Fold {f['fold']+1}: F1_final={m['f1_final']:.2f} | "              f"F1_q={m['f1_quality']:.4f} | F1_d={m['f1_defects']:.4f} | "              f"Epoch={f['best_epoch']}")else:    print("No CV results available.")

## 6. v2.1 Feature Impact Summary| Feature | Mechanism | Target ||---------|----------|--------|| **EMA** | Smooths training weights (decay=0.999) | Generalization || **Mixup** | Blends images + labels (alpha=0.2) | Imbalanced classes || **Quality-Aware Thresh** | Optimizes f1_final directly | Competition metric || **Temperature Scaling** | Calibrates probabilities (LBFGS) | Threshold quality || **68-dim Features** | PCA + depth + roughness | Geometric defects || **Progressive Resize** | 128→192→224px schedule | Training speed |